# Q4d · C7真实FPGA检测实验
原绿色局部特征和固定判别树均在PL执行；保留A/C2/C5。质量尚未验收，亮区误检及细血管仍可能残留。
固定20图×两尺寸×A/C2/C5/C7共160帧；C7分数阈值112，旧算法响应阈值28。两种阈值含义不同。无HDMI、无PS补掩膜。每框只运行一次，不要Run All。
关机后须从新的上传文件夹第1框开始，不能在没有前面上下文的新内核直接运行第5框。实验服务8090在第3框后启动，Jupyter管理页面仍9090。


## 1. 验证下载包并展开
应打印 PACKAGE_BYTES_PASS。只读验证通过后写入新的子文件夹；尚未访问 FPGA。

In [ ]:
from pathlib import Path
import hashlib, json, zipfile
if 'browser_package_root' in globals():
    raise RuntimeError('Already extracted; continue to the next cell')
bundle_path = Path('q4d_c7_experimental_r1.zip')
expected_bundle_sha256 = '9b0817670b34f1f0038ba545b56b318fff1ca74dae6c4cb6af026d50b14f5129'
if hashlib.sha256(bundle_path.read_bytes()).hexdigest() != expected_bundle_sha256:
    raise ValueError('Wrong download bytes; stop before importing or loading FPGA')
with zipfile.ZipFile(bundle_path) as archive:
    members = archive.infolist()
    names = [member.filename for member in members]
    manifest = json.loads(archive.read('q4d_manifest.json'))
    if len(names) != len(set(names)) or set(names) != set(manifest['artifacts']) | {'q4d_manifest.json'}:
        raise ValueError('Unexpected archive inventory')
    if len(members) > 200 or sum(member.file_size for member in members) > 128 * 1024 * 1024:
        raise ValueError('Unexpected package size')
    for member in members:
        path = Path(member.filename)
        if path.is_absolute() or '..' in path.parts or '\\' in member.filename or ':' in member.filename or member.is_dir():
            raise ValueError('Unsafe package member')
        if member.filename != 'q4d_manifest.json':
            data = archive.read(member)
            if {'bytes': len(data), 'sha256': hashlib.sha256(data).hexdigest()} != manifest['artifacts'][member.filename]:
                raise ValueError('Package mismatch: ' + member.filename)
    browser_package_root = (Path.cwd() / 'q4d_c7_experimental_r1').resolve()
    browser_package_root.mkdir(exist_ok=False)
    for member in members:
        destination = browser_package_root / member.filename
        if browser_package_root not in destination.resolve().parents:
            raise ValueError('Package path escaped directory')
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(archive.read(member))
print('PACKAGE_BYTES_PASS: paired Q4D and browser application bytes checked; FPGA untouched')


## 2. 检查环境并保留端口
应打印 BROWSER_PRECHECK_PASS 和实际版本。此步不加载FPGA。8090若被占用，保留报错，勿继续。

In [ ]:
import sys
if 'browser_context' in globals():
    raise RuntimeError('Already prepared; continue to the next cell')
sys.path.insert(0, str(browser_package_root))
from q4d_board import prepare_q4d, start, collect_q4d, export_q4d, finish
browser_context = prepare_q4d(browser_package_root, port=8090)


## 3. 加载一次并启动服务
应打印 BROWSER_SERVICE_READY。尚未进行图像收发。出现红色报错则保留内核和上下文，不重复加载。

In [ ]:
browser_start_status = await start(browser_context)
print(browser_start_status)


## 4. 自动核对160帧
逐帧打印Q4D_FRAME_SAVED，最后Q4D_COLLECTION_COMPLETE。20图、两尺寸、四算法输出SHA匹配仅代表实现一致，质量另审。期间不要在网页提交任务。


In [ ]:
q4d_collection = await collect_q4d(browser_context)
print(q4d_collection)


## 5. 浏览器视觉与参数对照
打开 http://192.168.2.99:8090/ 并Ctrl+F5，页眉须Q4d · C7。固定512、最终掩膜模式3、快速无损PNG。
按im0001、im0002、im0324顺序，每张先C5/阈值28运行，再C7/阈值112运行并截图，共6次。切换算法自动恢复对应推荐阈值，运行后检查硬件参数回读。观察亮边误检、细分支及跨亮区真血管。
保持网页前台：im0001/C7分别阈值96、128各运行一次；然后im0001/C5阈值20、36各运行一次。每次等图像更新。
再选im0001/C7、阈值112，点击预热3帧+测量20帧。保持网页前台至完成并截图。返回运行本状态框，预期completed=193、READY/IDLE、pending=0、无error或保留缓冲。额外点击记录照实保留，不为凑数重置。


In [ ]:
if 'browser_context' not in globals():
    raise RuntimeError('当前内核没有实验上下文，请回到运行第1–4框的同一Notebook；不要重复加载FPGA')
print(browser_context['manager'].status())


## 6. 导出完整记录
应出现Q4D_RESULTS_READY。下载本Notebook同目录q4d_results_bundle.zip；普通网页会话导出不能代替固定队列包。


In [ ]:
q4d_export_info = await export_q4d(browser_context, Path.cwd() / 'q4d_results_bundle.zip')


## 7. 安全结束并下载
应出现BROWSER_SERVICE_CLOSED_IDLE。Ctrl+S，再File → Download as → Notebook下载q4d_quality_start.ipynb，同时在文件列表下载q4d_results_bundle.zip。
红色报错或一分钟无进展：保留输出和内核，不重复执行、重新加载或释放失败缓冲。


In [ ]:
await finish(browser_context)
